# Chapter 11 — Building the Complete Agent

**Book alignment:** current Chapter 11 · internal demo `Stage 10`

The shared demo package calls this **Stage 10** internally. This chapter is integration only: it introduces no new conceptual mechanism.

**Question this notebook isolates:** Do the ten earned mechanisms compose without collapsing their boundaries, identities, budgets, or evidence semantics?


## Hypothesis

The accumulated architecture can repair the controlled broken repository in an isolated workspace, earn `VERIFIED_SUCCESS` from current protected evidence, and reject an evaluator shortcut even when naive pytest is green.


In [ ]:
from pathlib import Path
import sys
import tempfile


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "demo" / "agents-from-first-principles").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing demo/agents-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
DEMO_ROOT = REPO_ROOT / "demo" / "agents-from-first-principles"
TARGET = DEMO_ROOT / "examples" / "broken-parser"
sys.path.insert(0, str(DEMO_ROOT))

from first_principles_agent.app import (
    AgentApplication,
    run_protected_test_tampering_control,
)
from first_principles_agent.memory import MemoryStore

from first_principles_agent.verification import (
    IntegrityStatus,
    ProtectedPathVerifier,
    WorkspaceSnapshot,
)

## The final command

The CLI now defaults to the integrated Stage-10 application:

```bash
python -m first_principles_agent --repo examples/broken-parser --task "Fix pipe-delimited records"
```


## Experiment 1 - repair a real isolated workspace


In [ ]:
memory = MemoryStore()
app = AgentApplication(memory=memory)
run_root = Path(tempfile.mkdtemp(prefix="agents-book-stage10-"))
first = app.run("Fix pipe-delimited records", TARGET, workspace=run_root / "repair-1")

assert first.result == "VERIFIED_SUCCESS"
assert first.verification_verdict == "PASS"
assert first.verification_integrity == "CLEAN"
assert first.selected_candidate == "use-argument"
assert first.selection_metrics["oracle@N"] == 1
assert first.selection_metrics["selected_success@N"] == 1
assert first.selection_metrics["selection_gap@N"] == 0
assert first.revision_accepted is True
assert first.search_selected == "B1"
assert all(stage == "accepted" for stage in first.acceptance_stages)
assert "protected tests unchanged" in first.evidence
assert "split(delimiter)" in (Path(first.workspace) / "parser.py").read_text(
    encoding="utf-8"
)
assert 'return text.split(",")' in (TARGET / "parser.py").read_text(encoding="utf-8")
first.as_dict()

## Experiment 2 - verified memory changes a later run

The first successful run writes a reusable verified project fact. A second run remains independently correct but can use that memory instead of rediscovering the targeted test identity.


In [ ]:
second = app.run("Fix pipe-delimited records", TARGET, workspace=run_root / "repair-2")
assert first.memory_used is False
assert second.memory_used is True
assert second.result == "VERIFIED_SUCCESS"

## Experiment 3 - a green checker cannot launder test tampering

This is an external adversarial evaluation control, not an action available to the normal agent. It removes the protected pipe test, runs the real remaining suite, and then asks Stage 09 what that apparent success earns.


In [ ]:
tampered = run_protected_test_tampering_control(TARGET, workspace=run_root / "tampered")
assert tampered.naive_checker_passed is True
assert tampered.integrity == "VIOLATED"
assert tampered.final_verdict == "FAIL"
tampered

## Experiment 4 - regression for the verifier path-identity bug

The finished chapter records a real integration failure from the first adversarial run on Windows: one side named the protected asset `tests\test_parser.py`, another named it `tests/test_parser.py`, and both failed lookups collapsed to the same missing value.

The current verifier canonicalizes repository-relative paths **before** state identity and protected-path lookup. This cell replays that boundary directly.


In [ ]:
protected_test = "def test_pipe_delimiter(): pass\n"

windows_spelling = WorkspaceSnapshot.from_mapping(
    {
        r"tests\test_parser.py": protected_test,
        "parser.py": "broken",
    }
)
posix_spelling = WorkspaceSnapshot.from_mapping(
    {
        "tests/test_parser.py": protected_test,
        "parser.py": "broken",
    }
)

# Equivalent repository paths now produce the same state identity.
assert windows_spelling.state_id == posix_spelling.state_id

# Deleting the protected file must fail even when the baseline used Windows spelling.
deleted_protected = WorkspaceSnapshot.from_mapping({"parser.py": "broken"})
path_report = ProtectedPathVerifier(
    windows_spelling,
    protected_paths=("tests/test_parser.py",),
).check(deleted_protected)

assert path_report.status == IntegrityStatus.VIOLATED
assert path_report.violations == ("protected path deleted: tests/test_parser.py",)

# Conflicting aliases for the same canonical path fail closed rather than silently overwriting.
try:
    WorkspaceSnapshot.from_mapping(
        {
            r"tests\test_parser.py": "version A",
            "tests/test_parser.py": "version B",
        }
    )
except ValueError as exc:
    collision_rejected = "conflicting contents for canonical repository path" in str(
        exc
    )
else:
    collision_rejected = False

assert collision_rejected is True
{
    "equivalent_path_identity": windows_spelling.state_id.value,
    "deletion_detected": path_report.status.value,
    "canonical_collision_rejected": collision_rejected,
}

## The chapter's actual integration result: nine joints

A successful repair proves that the pieces can run. The stronger result is what happened **at the joints between mechanisms** when the full system was assembled.

| Joint | What collided | Resolution | Cost |
|---|---|---|---|
| `GoalContract` | Planning feasibility contract vs verification achievement contract, same name | Rename/alias the planning type `PlanContract` | One type rename |
| `EvidenceTier` | Search and verification independently defined the same vocabulary | Move to one shared enum | One shared type |
| `Verdict` | Critique union vs `PASS/FAIL/PARTIAL/UNKNOWN` | Rename the revision union `CritiqueVerdict` | One type rename |
| Candidate success | `verified_success=None` collapsed into `False`; labels do not exist before execution | Preserve tri-state values; compute `oracle@N` after isolated previews | One type hardening plus phase rule |
| Branch evidence | Search produces evidence for branch state `b1`; verifier adjudicates committed state `s1` | No special case: state binding rejects it | Zero; existing invariant held |
| Protected path | Stable policy was described as a precondition | Reject protected assets at **AUTHORIZATION**; reserve **PRECONDITION** for current-state assumptions | One stage correction |
| Memory promotion | One verified episode was being promoted directly to procedural memory | Store **episodic** memory; require repeated support for procedural promotion | One policy correction |
| Memory × search budget | Per-decision retrieval multiplied by branch factor | Retrieve once at the root under an explicit task-level applicability assumption | One documented assumption |
| Path identity | Windows and POSIX path spellings disagreed inside state/integrity lookup | Canonical repository-relative paths plus collision checks | One real verifier bug |

The useful pattern is that the joints that held were already joined by **explicit representation**: state identity, evidence type, named boundary stage. The joints that failed were places where two mechanisms merely shared a word or an informal convention.


## What was earned

Nothing new was added to the causal vocabulary. The existing mechanisms now compose into one runnable system: control, action acceptance, alternatives, revision, planning, temporal progress, capability selection, memory, isolated search, state-bound evidence and protected verification.

The model proposes. The runtime controls execution. The environment supplies evidence. The verifier decides what that evidence earns.
